# Advanced Pandas & Data Wrangling (Nigerian Market Data)

Beginner-friendly guided notebook.

In [ ]:

import pandas as pd
import numpy as np
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle
from reportlab.lib import colors


## 1. Read Datasets

In [ ]:

transactions = pd.read_csv("nigerian_market_transactions_1000.csv")
prices = pd.read_csv("nigerian_weekly_product_prices_1000.csv")

transactions.head()


## 2. Inspect Data

In [ ]:

transactions.info()
transactions.isna().sum()


## 3. Handle Missing Values

In [ ]:

transactions["quantity"] = transactions["quantity"].fillna(transactions["quantity"].median())
transactions["unit_price"] = transactions.groupby("product")["unit_price"].transform(lambda x: x.fillna(x.median()))
transactions["total_amount"] = transactions["quantity"] * transactions["unit_price"]


## 4. Remove Duplicates

In [ ]:

transactions = transactions.drop_duplicates()


## 5. Datetime Conversion

In [ ]:

transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"], errors="coerce")
prices["week_start_date"] = pd.to_datetime(prices["week_start_date"])


## 6. String Cleaning

In [ ]:

for df in [transactions, prices]:
    df["product"] = df["product"].str.lower().str.strip()

transactions["vendor"] = transactions["vendor"].str.strip().str.title()


## 7. Feature Engineering

In [ ]:

transactions["day_of_week"] = transactions["transaction_date"].dt.day_name()
transactions["week_start_date"] = transactions["transaction_date"].dt.to_period("W").dt.start_time


## 8. GroupBy & Aggregations

In [ ]:

transactions.groupby("market")["total_amount"].sum()


## 9. Merge Datasets

In [ ]:

merged_df = pd.merge(
    transactions,
    prices,
    how="left",
    on=["product", "market", "week_start_date"]
)

merged_df["price_diff"] = merged_df["unit_price"] - merged_df["avg_price"]
merged_df.head()


## 10. Export Clean Data to PDF

In [ ]:

pdf_data = merged_df.head(30)
table_data = [pdf_data.columns.tolist()] + pdf_data.values.tolist()

pdf = SimpleDocTemplate("clean_nigerian_market_data.pdf")
table = Table(table_data)

table.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
    ("GRID", (0,0), (-1,-1), 0.5, colors.grey),
    ("FONTSIZE", (0,0), (-1,-1), 8)
]))

pdf.build([table])
